In [44]:
import functools
import os
import sys
import json

from ultralytics import YOLO
import pandas as pd

sys.path.append("..")
sys.path.append("../..")
import ultralytics_validator_wrapper
from tflite_inference import TFLiteInferer, TFLiteMerged
from tflite_inference_input_adapter import TFLiteInferenceWithInputAdapter

pd.options.display.float_format = '{:.3f}'.format

%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [45]:
BASE_EXPERIMENT_DIR = "../results/cam3_revB_batch1_5_margin_15_rq_bg_wb_alif_clahe_box_01_dfl_03_2nd"
BASE_WEIGHTS_DIR = os.path.join(BASE_EXPERIMENT_DIR, "weights")
WEIGHTS_PTH = os.path.join(BASE_WEIGHTS_DIR, "best.pt")

HOLDOUT_PTH = " /home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/project_holdout.yaml"
VAL_PTH = "/home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/project.yaml"
POC_PTH = "/home/klymy/projects/researches/people_detection_pg2/quality_reduce_comparison/cam4_poc_15/project_pg.yaml"

if os.path.exists(os.path.join(BASE_EXPERIMENT_DIR, "inference_params.json")):
    with open(os.path.join(BASE_EXPERIMENT_DIR, "inference_params.json"), "r") as f:
        data = json.load(f)
    CONF_TH = float(data["conf_th"])
    IOU_TH = float(data["iou_th"])
else:
    print("No inference_params.json found, using default thresholds.")
    CONF_TH = 0.35
    IOU_TH = 0.50
    
IOU_TH = 0.35

args = dict(
    task="pose",
    device=[0],
    batch=128,
    verbose=False,
    conf=CONF_TH,
    iou=IOU_TH,
    imgsz=320,
    model="val",
    rect=False,
    single_cls=False,
)

In [46]:
model = YOLO(WEIGHTS_PTH)

In [47]:
inferer = TFLiteMerged(
    yolo_pth="combined_per_channel/combined_full_integer_quant.tflite",
    conf_th=CONF_TH,
    iou_th=IOU_TH,
    imgsz=320,
)

# inferer_adapter = TFLiteInferenceWithInputAdapter(
#     yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_saved_model/best_full_integer_quant.tflite"),
#     head_pth=os.path.join(BASE_WEIGHTS_DIR, "head_saved_model/head_float32.tflite"),
#     adapter_pth="../input_adapter/input_adapter_full_integer_quant.tflite",
#     conf_th=CONF_TH,
#     iou_th=IOU_TH,
#     imgsz=320,
# )

In [48]:
args["data"] = VAL_PTH

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)

# ultralytics_validator_wrapper.validate_model(
#     args,
#     model,
#     functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_adapter, args),
#     False,
# )

val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/labels/val.cache... 303 images, 1 backgrounds, 0 corrupt: 100%|██████████| 303/303 [00:00<?, ?it/s]


tst validation: 100%|██████████| 3/3 [00:11<00:00,  3.89s/it]


                   all        303       1504      0.814      0.861      0.806      0.257      0.933      0.963      0.972       0.97
   task    F1     P     R  MAP@50
0   box 0.837 0.814 0.861   0.806
1  pose 0.948 0.933 0.963   0.972


In [49]:
args["data"] = HOLDOUT_PTH

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)

# ultralytics_validator_wrapper.validate_model(
#     args,
#     model,
#     functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_adapter, args),
#     False,
# )

val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/labels/holdout.cache... 148 images, 5 backgrounds, 0 corrupt: 100%|██████████| 148/148 [00:00<?, ?it/s]


tst validation: 100%|██████████| 2/2 [00:05<00:00,  2.89s/it]


                   all        148        409      0.753      0.682      0.699      0.216      0.934      0.846      0.916      0.909
   task    F1     P     R  MAP@50
0   box 0.716 0.753 0.682   0.699
1  pose 0.888 0.934 0.846   0.916


In [50]:
args["data"] = POC_PTH

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)

# ultralytics_validator_wrapper.validate_model(
#     args,
#     model,
#     functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_adapter, args),
#     False,
# )

val: Scanning /home/klymy/projects/researches/people_detection_pg2/quality_reduce_comparison/cam4_poc_15/labels/val.cache... 191 images, 5 backgrounds, 0 corrupt: 100%|██████████| 191/191 [00:00<?, ?it/s]


tst validation: 100%|██████████| 2/2 [00:07<00:00,  3.69s/it]


                   all        191        417      0.592      0.616      0.557      0.171      0.709      0.739      0.741      0.726
   task    F1     P     R  MAP@50
0   box 0.604 0.592 0.616   0.557
1  pose 0.723 0.709 0.739   0.741
